<a href="https://colab.research.google.com/github/HazemmoAlsady/AWN_Graduation_Project/blob/main/classification_model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d kiva/data-science-for-good-kiva-crowdfunding

Dataset URL: https://www.kaggle.com/datasets/kiva/data-science-for-good-kiva-crowdfunding
License(s): CC0-1.0
data-science-for-good-kiva-crowdfunding.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
!unzip /content/data-science-for-good-kiva-crowdfunding.zip -d dataset

Archive:  /content/data-science-for-good-kiva-crowdfunding.zip
replace dataset/kiva_loans.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset/kiva_loans.csv")
df


In [ ]:
df.isnull().sum()

In [ ]:
df = df[["use", "sector"]]

df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)


In [ ]:
import re
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df["request_text"] = df["use"].apply(clean_text)

In [ ]:
def map_label(sector):
    if sector in ["Agriculture", "Retail", "Transportation", "Services",
                  "Manufacturing", "Clothing", "Construction", "Wholesale"]:
        return "مالي"
    elif sector == "Education":
        return "تعليمي"
    elif sector == "Health":
        return "صحي"
    elif sector == "Housing":
        return "سكني"
    elif sector == "Food":
        return "غذاء"
    else:
        return "مالي"

df["label"] = df["sector"].apply(map_label)
df = df[["request_text", "label"]]

print("توزيع الفئات قبل التوازن:")
print(df["label"].value_counts())

In [ ]:
min_count = df["label"].value_counts().min()
df_balanced = df.groupby("label").apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)

print("\nتوزيع الفئات بعد التوازن:")
print(df_balanced["label"].value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_balanced["request_text"])


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df_balanced["label"])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred ))

In [ ]:
test_text = ["To buy seasonal, fresh fruits to sell."]


test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)
print("Prediction:", le.inverse_transform(prediction))

In [ ]:
test_text = ["I need money to buy books for school"]

test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)

print(le.inverse_transform(prediction))

In [ ]:
tests = [
    "I want to buy food for my family",
    "Need money for school fees",
    "Medical treatment for my child",
    "Build a small house",
    "Start a small business"
]

test_vec = vectorizer.transform(tests)
preds = model.predict(test_vec)

for text, pred in zip(tests, preds):
    print(text, "→", le.inverse_transform([pred])[0])

In [ ]:
!pip install deep-translator

In [ ]:
from deep_translator import GoogleTranslator

def predict_arabic(text_ar):
    # 1️⃣ ترجمة عربي → إنجليزي
    text_en = GoogleTranslator(source='auto', target='en').translate(text_ar)

    # 2️⃣ تحويل لـ vector
    vec = vectorizer.transform([text_en])

    # 3️⃣ prediction
    pred = model.predict(vec)

    # 4️⃣ رجوع label عربي
    result = le.inverse_transform(pred)[0]

    return result, text_en

In [ ]:
arabic_tests = [
    "عايز فلوس أبدأ مشروع صغير",
    "محتاج أشتري أكل لأسرتي",
    "عايز أدفع مصاريف المدرسة",
    "محتاج علاج لابني",
    "عايز أبني بيت صغير",
    "عايز أشتري بضاعة للمحل",
    "محتاج أجيب خضار وفاكهة للبيع",
    "عايز أوسع المشروع بتاعي",
    "محتاج أصلح البيت",
    "عايز أشتري أدوية",
    "محتاج أبدأ مشروع تربية مواشي",
    "عايز أفتح كشك صغير",
    "محتاج فلوس للجامعة",
    "عايز أجيب أدوات مدرسية",
    "محتاج علاج عملية جراحية",
    "عايز أشتري مواد بناء",
    "محتاج أبدأ مشروع أكل",
    "عايز أشتري ماكينة خياطة",
    "محتاج فلوس لمصاريف البيت",
    "عايز أفتح مطعم صغير"
]

In [ ]:
for text in arabic_tests:
    result, translated = predict_arabic(text)
    print(f"AR: {text}")
    print(f"EN: {translated}")
    print(f"Prediction: {result}")
    print("-" * 50)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/models", exist_ok=True)

In [ ]:
import joblib

joblib.dump(model, "/content/drive/MyDrive/graduation_project/models/model.pkl")
joblib.dump(vectorizer, "/content/drive/MyDrive/graduation_project/models/vectorizer.pkl")
joblib.dump(le, "/content/drive/MyDrive/graduation_project/models/label_encoder.pkl")